In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import Normalize

from sklearn.metrics import roc_curve, auc, f1_score
from scipy.stats import norm
sys.path.append("../")

plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["font.size"] = 9
plt.rcParams["pdf.fonttype"] = 42  # embed TrueType instead of Type 3 fonts

## AUC

In [2]:
# Base directory
base_dir = "..\\Anomaly_Scores"

# Jammer names
jammers = ["barrage", "deceptive", "pilot", "random_hop", "sweep"]

# Models: (label file, score file)
models = {
    "Supervised (Including)": (
        "Labels_Supervised.npy",
        "Scores_Supervised_including_model.npy"
    ),
    "Supervised (Excluding)": (
        "Labels_Supervised.npy",
        "Scores_Supervised_excluding_model.npy"
    ),
    "Unsupervised": (
        "Labels_Unsupervised.npy",
        "Scores_Unsupervised_model.npy"
    )
}

In [3]:

for jammer in jammers:

    title = jammer.replace("_", " ").title()
    print(f"\n--------{title} Evaluation--------")

    jammer_dir = os.path.join(base_dir, jammer)

    for model_name, (label_file, score_file) in models.items():

        # Load labels and scores
        y_true = np.load(os.path.join(jammer_dir, label_file)).ravel()
        y_labels = (y_true != "Normal").astype(int)
        y_score = np.load(os.path.join(jammer_dir, score_file)).ravel()
        
        # Calculate ROC curve and AUC
        fpr, tpr, _ = roc_curve(y_labels, y_score)
        roc_auc = auc(fpr, tpr)

        print(f"{model_name}: auc= {roc_auc}")



--------Barrage Evaluation--------
Supervised (Including): auc= 0.997288
Supervised (Excluding): auc= 0.989194
Unsupervised: auc= 0.926104

--------Deceptive Evaluation--------
Supervised (Including): auc= 0.9400170000000001
Supervised (Excluding): auc= 0.7513369999999999
Unsupervised: auc= 0.8862939999999999

--------Pilot Evaluation--------
Supervised (Including): auc= 0.9345400000000001
Supervised (Excluding): auc= 0.747473
Unsupervised: auc= 0.8785559999999999

--------Random Hop Evaluation--------
Supervised (Including): auc= 0.989824
Supervised (Excluding): auc= 0.9543319999999998
Unsupervised: auc= 0.9683699999999998

--------Sweep Evaluation--------
Supervised (Including): auc= 0.9907239999999999
Supervised (Excluding): auc= 0.98016
Unsupervised: auc= 0.973374


## F1 Score

In [4]:
# Base directory
base_dir = "..\\Anomaly_Scores"

# Jammer names
jammers = ["barrage", "deceptive", "pilot", "random_hop", "sweep"]

# Models: (label file, score file)
models = {
    "Supervised (Including)": (
        "Labels_Supervised.npy",
        "Scores_Supervised_including_model.npy"
    ),
    "Supervised (Excluding)": (
        "Labels_Supervised.npy",
        "Scores_Supervised_excluding_model.npy"
    ),
    "Unsupervised": (
        "Labels_Unsupervised.npy",
        "Scores_Unsupervised_model.npy"
    )
}

In [5]:

def calculate_f1(labels, scores, threshold):
    labels = np.asarray(labels)
    scores = np.asarray(scores)

    predictions = (scores >= threshold).astype(int)

    return f1_score(labels, predictions)


def get_threshold(validation_scores, percentile=0.9):
    return np.quantile(validation_scores, percentile)

In [6]:

for jammer in jammers:

    title = jammer.replace("_", " ").title()
    print(f"\n--------{title} Evaluation--------")

    jammer_dir = os.path.join(base_dir, jammer)

    for model_name, (label_file, score_file) in models.items():
        if model_name.startswith("Supervised"):
            
            y_true = np.load(os.path.join(jammer_dir, label_file)).ravel()
            y_labels = (y_true != "Normal").astype(int)
            y_score = np.load(os.path.join(jammer_dir, score_file)).ravel()
            
            f1 = calculate_f1(y_labels, y_score, threshold=0.5)

            print(f"{model_name}: F1 Score = {f1}")
        
        elif model_name.startswith("Unsupervised"):
            val_scores = np.load(os.path.join(base_dir, "Validation_Scores.npy"))
            mu = np.mean(val_scores)
            std = np.sqrt(np.var(val_scores))
            val_scores_n = (val_scores - mu) / std
            th = get_threshold(val_scores_n, percentile=0.85)

            y_true = np.load(os.path.join(jammer_dir, label_file)).ravel()
            y_labels = (y_true != "Normal").astype(int)
            y_score = np.load(os.path.join(jammer_dir, score_file)).ravel()
            
            f1 = calculate_f1(y_labels, y_score, threshold=th)
            print(f"{model_name}: F1 Score = {f1}")


--------Barrage Evaluation--------
Supervised (Including): F1 Score = 0.9726027397260274
Supervised (Excluding): F1 Score = 0.9442815249266863
Unsupervised: F1 Score = 0.8369462770970783

--------Deceptive Evaluation--------
Supervised (Including): F1 Score = 0.8817891373801917
Supervised (Excluding): F1 Score = 0.44036697247706424
Unsupervised: F1 Score = 0.7395301327885597

--------Pilot Evaluation--------
Supervised (Including): F1 Score = 0.8841657810839533
Supervised (Excluding): F1 Score = 0.529839883551674
Unsupervised: F1 Score = 0.7051416579223505

--------Random Hop Evaluation--------
Supervised (Including): F1 Score = 0.9624505928853755
Supervised (Excluding): F1 Score = 0.8817204301075269
Unsupervised: F1 Score = 0.8720292504570384

--------Sweep Evaluation--------
Supervised (Including): F1 Score = 0.9644970414201184
Supervised (Excluding): F1 Score = 0.9263370332996973
Unsupervised: F1 Score = 0.8842676311030742
